# PixelBoost — Colab T4 Turbo (optional GPU accelerator)

Run this notebook to turn a free Google Colab **GPU (T4)** session into a
PixelBoost AI worker. While it is running, all "AI Plus" upscales from the
website will be routed to it automatically (second-ish speeds instead of
20-90s of free-CPU inference). If the notebook closes, the backend falls back
to the free HuggingFace CPU worker — nothing breaks.

**Cost:** free (Colab), limited to ~3-4 h per session. Re-run to refresh.

**Before starting:** set `PIXELBOOST_COLAB_SECRET` on the Render backend and
paste the **same** value below.

Stability notes:
- The notebook re-registers itself every 30 s so the backend knows it's alive.
- The tunnel auto-heals: if the public URL changes, the notebook re-registers
  with the new one automatically.
- Model weights are cached on the Colab disk, so re-runs are fast.

In [ ]:
# --- 1) Installers ---
!pip install -q fastapi uvicorn cloudflared huggingface_hub
!pip list | grep -Ei 'fastapi|uvicorn|cloudflared|torch'

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

In [ ]:
# --- 3) Config (edit these) ---
BACKEND_URL = "https://pixelboost-backend-q659.onrender.com"  # your backend
SHARE_SECRET = "CHANGE_ME"  # must equal PIXELBOOST_COLAB_SECRET on the backend
WORKER_MODE = "ai-plus"  # which mode this worker serves (ai-fast | ai-plus | anime)
LOCAL_PORT = 8080

In [ ]:
# --- 2) Real-ESRGAN on GPU (inline arch, same as the HF Space) ---
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from urllib.request import urlretrieve
from concurrent.futures import ThreadPoolExecutor

WEIGHTS_DIR = "/content/pixelboost_weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

class SRVGGNetCompact(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=32, upscale=4):
        super().__init__()
        self.upscale = upscale
        self.body = nn.ModuleList()
        self.body.append(nn.Conv2d(num_in_ch, num_feat, 3, 1, 1))
        self.body.append(nn.PReLU(num_parameters=num_feat))
        for _ in range(num_conv):
            self.body.append(nn.Conv2d(num_feat, num_feat, 3, 1, 1))
            self.body.append(nn.PReLU(num_parameters=num_feat))
        self.body.append(nn.Conv2d(num_feat, num_out_ch * upscale * upscale, 3, 1, 1))
        self.upsampler = nn.PixelShuffle(upscale)
    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        out = self.upsampler(out)
        return out + F.interpolate(x, scale_factor=self.upscale, mode="nearest")

class ResidualDenseBlock(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)
    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, upscale=4):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body = nn.ModuleList([RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.body[0](feat)
        for block in self.body[1:]:
            body_feat = block(body_feat)
        feat = self.conv_body(feat + body_feat)
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.conv_hr(feat)
        return self.conv_last(self.lrelu(feat))

MODEL_SPECS = {
    "ai-fast": {"label": "realesr-general-x4v3", "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth", "arch": "srvgg", "tile": 512},
    "ai-plus": {"label": "RealESRGAN_x4plus", "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1.0/RealESRGAN_x4plus.pth", "arch": "rrdb", "blocks": 23, "tile": 512},
    "anime": {"label": "RealESRGAN_x4plus_anime_6B", "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth", "arch": "rrdb", "blocks": 6, "tile": 512},
}

def load_model(mode):
    spec = MODEL_SPECS[mode]
    dest = os.path.join(WEIGHTS_DIR, f"{mode}.pth")
    if not os.path.exists(dest):
        urlretrieve(spec["url"], dest)
    if spec["arch"] == "srvgg":
        model = SRVGGNetCompact()
    else:
        model = RRDBNet(num_block=spec["blocks"])
    state = torch.load(dest, map_location="cpu")
    key = "params_ema" if "params_ema" in state else "params"
    model.load_state_dict(state[key], strict=True)
    model.to(DEVICE).eval()
    return model

MODEL = load_model(WORKER_MODE)
print("model ready:", MODEL_SPECS[WORKER_MODE]["label"])

In [ ]:
import io
import numpy as np
from PIL import Image

NATIVE = 4

@torch.inference_mode()
def infer(img_bytes: bytes, scale: int) -> Image.Image:
    image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    arr = np.asarray(image, dtype=np.float32) / 255.0
    h, w, _ = arr.shape
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    tile = MODEL_SPECS[WORKER_MODE]["tile"]
    pad = 16
    out_h, out_w = h * NATIVE, w * NATIVE
    out = torch.zeros((1, 3, out_h, out_w), device=DEVICE, dtype=torch.float32)

    for ty in range(max(1, (h + tile - 1) // tile)):
        for tx in range(max(1, (w + tile - 1) // tile)):
            y0, x0 = ty * tile, tx * tile
            y1, x1 = min(y0 + tile, h), min(x0 + tile, w)
            py0, px0 = max(0, y0 - pad), max(0, x0 - pad)
            py1, px1 = min(h, y1 + pad), min(w, x1 + pad)
            patch = t[:, :, py0:py1, px0:px1]
            up = MODEL(patch).clamp_(0, 1)
            ct, cl = (y0 - py0) * NATIVE, (x0 - px0) * NATIVE
            ch, cw = (y1 - y0) * NATIVE, (x1 - x0) * NATIVE
            out[:, :, y0*NATIVE:y1*NATIVE, x0*NATIVE:x1*NATIVE] = up[:, :, ct:ct+ch, cl:cl+cw]

    out_img = (out.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    result = Image.fromarray(out_img)
    target = (w * scale, h * scale)
    if result.size != target:
        result = result.resize(target, Image.Resampling.LANCZOS)
    return result

In [ ]:
# --- 4) FastAPI server + tunnel + auto-registration with the backend ---
import asyncio
import threading
import subprocess
import httpx
import uvicorn
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import Response

app = FastAPI(title="pixelboost-worker")

@app.post("/upscale")
async def upscale_endpoint(file: UploadFile = File(...), scale: int = Form(4),
                           mode: str = Form(WORKER_MODE), face: str = Form("0")):
    raw = await file.read()
    result = infer(raw, scale)
    buf = io.BytesIO()
    result.save(buf, format="PNG")
    return Response(content=buf.getvalue(), media_type="image/png")

current_public_url = {"url": None}

def run_tunnel():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{LOCAL_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        line = line.strip()
        if "trycloudflare.com" in line and current_public_url["url"] is None:
            url = next((tok for tok in line.split() if "trycloudflare.com" in tok), None)
            if url:
                current_public_url["url"] = url.rstrip("/")
                print("\n=== PUBLIC URL:", current_public_url["url"], "===\n", flush=True)
        if current_public_url["url"]:
            print("tunnel:", line, flush=True)

def register_loop():
    while True:
        try:
            url = current_public_url["url"]
            if url:
                r = httpx.post(
                    f"{BACKEND_URL.rstrip('/')}/workers/register",
                    data={"url": url, "secret": SHARE_SECRET, "mode": WORKER_MODE},
                    timeout=20,
                )
                print("registering:", r.status_code, r.text[:120], flush=True)
        except Exception as e:
            print("register loop error:", e, flush=True)
        time.sleep(30)

threading.Thread(target=run_tunnel, daemon=True).start()
time.sleep(12)
threading.Thread(target=register_loop, daemon=True).start()

# Run the API server (blocking in this cell; stop this cell to shut down).
uvicorn.run(app, host="0.0.0.0", port=LOCAL_PORT, log_level="info")

## What's happening
- The API server is now listening on the local port.
- A Cloudflare tunnel exposes it at `https://<id>.trycloudflare.com`.
- The notebook re-registers that URL with your backend every 30 s.
- Backend AI jobs now route here first (fast), falling back to HF CPU if
  this session dies.

Keep this transcript open to keep Colab running. Stop the cell above to
gracefully stop offering GPU service.